# Pipeline, Tratamento e Engenharia de Dados

Este notebook apresenta e documenta as etapas de tratamento, processamento, validação e organização dos dados desenvolvidas durante o projeto.
O objetivo é demonstrar como a engenharia de dados contribui para a construção de uma base consistente e organizada, fornecendo os dados necessários para as etapas de geração de características e classificação multimodal.

## 1) Visão geral do pipeline de dados

O pipeline de dados organiza as etapas necessárias para transformar os arquivos originais em uma base estruturada, validada e preparada para as etapas posteriores de Machine Learning.

O fluxo está organizado da seguinte maneira:

1. **Dados originais:** os arquivos CSV são armazenados no diretório `data/raw/`.
2. **Limpeza inicial:** o módulo `cleaner.py` realiza operações de limpeza e padronização, gerando os arquivos tratados no diretório `data/clean/`.
3. **Processamento:** o módulo `processor.py` utiliza as configurações de `sources.yaml` para carregar os dados, aplicar transformações e preparar a estrutura do dataset.
4. **Validação:** o módulo `validator.py` verifica a integridade e a qualidade dos dados conforme as regras estabelecidas.
5. **Persistência:** após o processamento e a validação, a base é armazenada no formato Parquet, no diretório `data/processed/`.
6. **Divisão dos dados:** o módulo `splitter.py` organiza os registros em conjuntos de treinamento, validação e teste, seguindo os parâmetros definidos em `models.yaml`.

É importante destacar que a limpeza inicial e o pipeline principal são executados por módulos distintos. O `cleaner.py` gera o arquivo de entrada tratado, enquanto o `pipeline.py` coordena o processamento, a validação e a divisão dos dados. Essa separação permite manter cada responsabilidade independente, facilitando a manutenção, a rastreabilidade e a evolução do processo de preparação dos dados.

## 2) Organização dos dados e estrutura de diretórios

### Estrutura utilizada

```text
data/
├── raw/
│   └── chamados.csv
│
├── clean/
│   └── chamados-clean.csv
│
└── processed/
    └── chamados.parquet

reports/
└── data/
    └── quality.json
```

### Finalidade dos diretórios

- **`data/raw/`**: armazena os arquivos originais recebidos pelo projeto. Esses dados são a fonte inicial do processo e devem ser preservados.
- **`data/clean/`**: armazena os arquivos após a etapa inicial de limpeza e padronização realizada pelo `cleaner.py`.
- **`data/processed/`**: recebe os dados processados e validados pelo pipeline, armazenados no formato Parquet.
- **`reports/data/`**: armazena relatórios relacionados à qualidade dos dados, permitindo consultar os resultados das verificações realizadas durante o processamento.

### Fluxo de armazenamento

O fluxo começa com os arquivos originais em `data/raw/`. Após a limpeza inicial, uma versão tratada é disponibilizada em `data/clean/`.

Em seguida, o processamento principal utiliza esse arquivo como entrada, aplica as transformações configuradas e executa as verificações de qualidade. Quando o processamento é concluído, o conjunto de dados é salvo em `data/processed/`, enquanto o relatório de qualidade é armazenado separadamente.

## 3) Limpeza inicial dos dados — `cleaner.py`

A etapa de limpeza inicial tem como objetivo reduzir inconsistências nos arquivos de entrada e preparar os dados para o processamento posterior.

Essa etapa é implementada em `src/data/cleaner.py` e atua sobre os arquivos CSV armazenados no diretório `data/raw/`, gerando versões tratadas no diretório `data/clean/`.

### Leitura dos arquivos

O primeiro cuidado é garantir que os arquivos CSV sejam carregados corretamente.

A função `careful_load_csv()` tenta realizar a leitura utilizando diferentes codificações de caracteres, além de testar diferentes separadores, como vírgula e ponto e vírgula.

O código também reconhece determinados marcadores textuais, como `nan`, `?`, `null` e `none`, interpretando-os como valores ausentes durante a leitura.

Essa estratégia ajuda a evitar problemas comuns na importação de arquivos, como caracteres acentuados interpretados incorretamente ou colunas carregadas de forma inadequada.

### Padronização e limpeza

Após a leitura, a função `clean_dataframe()` aplica uma sequência de operações:

1. **Cópia do DataFrame:** cria uma cópia dos dados para realizar as alterações sem modificar diretamente o objeto original.
2. **Remoção de linhas completamente vazias:** elimina registros que não possuem nenhuma informação.
3. **Remoção de duplicatas:** remove linhas repetidas, reduzindo a redundância dos registros.
4. **Padronização dos nomes das colunas:** remove acentos, caracteres de marcação, espaços excedentes e prefixos numéricos iniciais, além de converter os nomes para letras minúsculas.
5. **Limpeza de valores textuais:** remove espaços em branco no início e no final dos textos e transforma strings vazias em valores nulos.
6. **Conversão de tipos de dados:** tenta identificar colunas que representam números ou datas e realizar a conversão quando uma proporção suficiente dos valores permite essa interpretação.

As conversões são realizadas por meio de tentativas automáticas. Para valores numéricos, o código considera uma conversão bem-sucedida quando pelo menos 90% dos registros da coluna podem ser interpretados como números. Caso contrário, tenta interpretar os valores como datas, utilizando como referência pelo menos 80% dos valores não nulos.

### Identificação de valores ausentes

O código também possui a função `get_null_amount()`, utilizada para apresentar informações sobre a quantidade e a proporção de valores ausentes em cada coluna.

É importante destacar que essa função apresenta um relatório dos valores ausentes, mas não realiza, por si só, a remoção ou o preenchimento desses valores.

### Geração dos arquivos limpos

Por meio da função `main()`, o script percorre os arquivos CSV disponíveis em `data/raw/`, executa as operações de limpeza e salva os resultados em `data/clean/`.

Os arquivos gerados recebem o sufixo `-clean`, permitindo diferenciá-los dos arquivos originais.

Caso ocorra um erro durante o processamento de um arquivo, o script registra a ocorrência e continua tentando processar os demais arquivos disponíveis.

### Importância dessa etapa para o Machine Learning

A limpeza inicial contribui para a qualidade dos dados que serão utilizados posteriormente no Machine Learning.

A padronização de nomes, a remoção de registros vazios e duplicados e a identificação de possíveis inconsistências reduzem problemas que poderiam comprometer as etapas seguintes de processamento.

Entretanto, essa etapa não representa toda a preparação dos dados para o Machine Learning. Regras mais específicas, como conversões controladas de tipos, validação de campos obrigatórios e verificação das categorias, são tratadas posteriormente pelo pipeline principal.

## 4) Processamento e transformação dos dados — `processor.py` e `sources.yaml`

Após a limpeza inicial, os dados passam por uma etapa de processamento mais específica, implementada em `src/data/processor.py`.

Essa etapa utiliza o arquivo `src/config/sources.yaml` como referência para determinar quais dados serão carregados, quais transformações devem ser aplicadas, quais colunas devem ser removidas e quais critérios de qualidade precisam ser atendidos.

### Configuração da entrada e saída

No arquivo `sources.yaml`, são definidos os caminhos utilizados pelo processamento:

- **Entrada:** `data/clean/chamados-clean.csv`
- **Saída:** `data/processed/chamados.parquet`

Também são especificados os parâmetros de leitura do CSV, como o separador utilizado e a codificação dos caracteres.

### Conversão e padronização dos tipos de dados

Uma das principais responsabilidades do `processor.py` é garantir que determinadas colunas tenham tipos de dados adequados.

Durante a leitura do CSV, as colunas configuradas para transformação são inicialmente carregadas como texto (`string`). Em seguida, a função `cast_column()` realiza as conversões definidas no `sources.yaml`.

Entre as conversões configuradas estão:

| Coluna | Tipo definido | Finalidade |
|---|---|---|
| `ticket_id` | String | Identificação do chamado |
| `sla_hours` | Float64 | Representação numérica do SLA |
| `resolution_time_hours` | Float64 | Representação numérica do tempo de resolução |
| `sentiment_score` | Float64 | Representação numérica do sentimento |
| `num_words` | Float64 | Representação numérica da quantidade de palavras |
| `image_path` | String | Caminho da imagem associada |
| `title` | String | Texto do título do chamado |
| `description` | String | Texto da descrição |
| `target_category` | String | Categoria associada ao chamado |

A função `cast_column()` também possui mecanismos para lidar com valores incompatíveis com o tipo esperado.

Existem duas estratégias possíveis:

- `raise`: interrompe o processamento quando encontra valores incompatíveis.
- `coerce`: transforma valores incompatíveis em valores nulos.

Na configuração atual, como as regras não especificam outra estratégia, o comportamento padrão é `raise`.

### Remoção de colunas desnecessárias

Além das conversões, o arquivo `sources.yaml` define uma lista de colunas que devem ser removidas durante o processamento, por meio da configuração `drop_cols`.

Essa operação reduz a quantidade de informações mantidas e permite concentrar a base nos atributos definidos para as etapas posteriores.

Antes de realizar as transformações, o código também verifica se as colunas configuradas para conversão e remoção estão presentes no DataFrame.

### Execução do processamento

A função `process_dataset()` coordena o fluxo principal dessa etapa:

1. Carrega as configurações do arquivo `sources.yaml`.
2. Identifica os caminhos dos arquivos de entrada e saída.
3. Realiza a leitura do CSV ou Parquet de entrada.
4. Aplica as conversões de tipos e remove as colunas configuradas.
5. Encaminha o resultado para a validação da qualidade dos dados.
6. Caso a validação e a gravação sejam concluídas corretamente, salva o conjunto processado no formato Parquet.

O processamento também possui uma opção de execução chamada `dry_run`, que permite executar as operações e as verificações sem gravar o arquivo final.

### Persistência em Parquet

Após a transformação e a validação, o conjunto de dados é armazenado no formato Parquet.

Esse formato utiliza uma organização colunar, sendo adequado para armazenar dados estruturados e preservar os tipos definidos durante o processamento.

O código realiza a gravação inicialmente em um arquivo temporário. Somente depois de concluir a escrita com sucesso, esse arquivo substitui o destino final.

### Importância para a Engenharia de Dados

Essa etapa transforma um arquivo previamente limpo em um conjunto de dados com estrutura e tipos mais bem definidos.

A conversão controlada de tipos reduz ambiguidades na interpretação dos valores, enquanto a remoção de colunas permite organizar os atributos que serão utilizados posteriormente.

A validação antes da persistência também funciona como uma proteção: o novo arquivo Parquet só é gravado depois que as verificações de qualidade são concluídas.

Com isso, o projeto passa a trabalhar com uma base processada, estruturada e preparada para as próximas etapas do pipeline de dados.

## 5) Validação da qualidade dos dados — `validator.py`

Após as transformações realizadas pelo `processor.py`, os dados passam por uma etapa de validação, implementada em `src/data/validator.py`.

O objetivo dessa etapa é verificar se o conjunto de dados atende aos critérios de qualidade definidos no projeto, identificando problemas que possam comprometer sua utilização nas etapas posteriores.

As regras de validação são configuradas no arquivo `src/config/sources.yaml`, permitindo ajustar os critérios sem modificar diretamente o código do validador.

### Verificação da estrutura do dataset

O validador verifica se as colunas obrigatórias estão presentes no conjunto de dados.

Na configuração atual, são exigidas 11 colunas:

- `ticket_id`
- `target_category`
- `title`
- `description`
- `image_path`
- `sla_hours`
- `sentiment_score`
- `num_words`
- `device_type`
- `os`
- `network_type`

### Verificação de valores ausentes e duplicados

O projeto define quatro colunas que não devem conter valores ausentes:

- `ticket_id`
- `target_category`
- `description`
- `image_path`

O validador considera como ausentes tanto valores nulos quanto strings vazias ou compostas apenas por espaços.

Outra regra importante é a unicidade dos identificadores. A coluna `ticket_id` é definida como chave única, evitando que diferentes registros utilizem o mesmo identificador.

O código também identifica linhas inteiramente duplicadas. Nesse caso, a ocorrência é registrada como aviso, sem necessariamente interromper o processamento.

### Verificação dos tipos e intervalos numéricos

Além de verificar a estrutura, o sistema confere se determinadas colunas possuem tipos numéricos e se seus valores respeitam os intervalos configurados.

As regras atuais incluem:

| Coluna | Regra de validação |
|---|---|
| `sla_hours` | Valor maior ou igual a zero |
| `num_words` | Valor maior ou igual a zero |
| `sentiment_score` | Valor entre -1 e 1 |

O validador também identifica valores numéricos inválidos e valores infinitos.

Essas verificações ajudam a evitar que valores incompatíveis com o significado esperado das variáveis sejam aceitos sem identificação.

### Validação das categorias

A coluna `target_category` possui um conjunto de valores permitidos, correspondente às seis categorias de chamados do projeto:

- Software;
- Rede;
- Hardware;
- RH;
- Acesso;
- Financeiro.

O validador verifica se os registros utilizam categorias pertencentes a essa lista.

### Verificação das imagens

Como o projeto também trabalha com informações visuais, o validador possui verificações específicas para os arquivos de imagem associados aos chamados.

Entre os procedimentos realizados estão:

- Verificar se o caminho informado corresponde a um arquivo existente.
- Tentar decodificar e carregar o conteúdo da imagem.
- Calcular o hash SHA-256 dos arquivos.
- Registrar as dimensões das imagens.
- Identificar caminhos diferentes que correspondem ao mesmo conteúdo.

O cálculo do hash permite identificar imagens com conteúdo idêntico, mesmo quando seus nomes ou caminhos são diferentes.

Essa informação também pode ser aproveitada pela etapa de divisão dos dados, que utiliza os hashes das imagens para auxiliar na organização dos grupos.

### Verificações textuais e limite de valores ausentes

O `validator.py` também possui funcionalidades para analisar colunas textuais, identificando valores repetidos e textos idênticos associados a diferentes categorias.

Outra regra configurada é o limite de valores ausentes (`limite_nulos`), definido como `0.95`, com severidade `warning`. Isso significa que colunas com proporção de valores ausentes superior a 95% geram avisos no relatório, sem necessariamente reprovar a validação.

### Geração do relatório de qualidade

Durante a validação, o projeto reúne informações sobre as verificações realizadas e gera um relatório em formato JSON.

O relatório reúne informações como:

- Quantidade de registros e colunas.
- Tipos de dados e quantidade de valores ausentes por coluna.
- Quantidade de valores únicos.
- Problemas identificados durante as verificações.
- Informações sobre os arquivos de imagem.
- Quantidade de erros e avisos.
- Resultado geral da validação.

O relatório é gravado de maneira controlada, utilizando um arquivo temporário antes de substituir o arquivo de destino. Esse procedimento ajuda a preservar a versão anterior caso ocorra uma falha durante a gravação.

### Tratamento dos resultados da validação

Ao finalizar as verificações, o sistema contabiliza separadamente erros e avisos.

Quando existem erros e a opção `fail_on_error` está habilitada, o validador lança uma exceção chamada `ValidationError`, interrompendo o fluxo de processamento.

Os avisos, por outro lado, são registrados e apresentados sem necessariamente impedir a continuidade.

### Importância da validação para o Machine Learning

A validação funciona como uma etapa de controle da qualidade dos dados antes de sua utilização posterior.

Ao verificar estrutura, integridade, tipos, categorias e arquivos de imagem, o projeto reduz o risco de inconsistências passarem despercebidas e afetarem as etapas seguintes.

Dessa forma, o conjunto de dados processado não é avaliado apenas pela sua existência, mas também pelo atendimento aos critérios de qualidade definidos para o projeto.

## 6) Divisão dos dados — `splitter.py`

Após o processamento e a validação, o dataset é dividido em três conjuntos: treinamento, validação e teste. Essa etapa é coordenada pelo módulo `splitter.py`, que utiliza as configurações definidas em `models.yaml`.

### Estratégia de divisão

A configuração atual estabelece as seguintes proporções:

- **Treinamento:** 70% dos dados.
- **Validação:** 15% dos dados.
- **Teste:** 15% dos dados.

A estratégia selecionada é `group`, utilizando a coluna `image_path` como referência para agrupar os registros. Entretanto, o agrupamento é realizado pelo conteúdo das imagens, e não apenas pelo caminho dos arquivos.

Para isso, o código calcula o hash SHA-256 de cada imagem. Dessa forma, imagens com conteúdo idêntico recebem o mesmo identificador de grupo, mesmo que estejam armazenadas em caminhos diferentes.

Essa abordagem evita que chamados associados à mesma imagem sejam distribuídos entre treinamento, validação e teste, reduzindo o risco de vazamento de informações (*data leakage*).

### Busca pela melhor divisão

O módulo utiliza `GroupShuffleSplit`, do Scikit-learn, para gerar possíveis divisões respeitando os grupos identificados.

Como essa estratégia não realiza estratificação automática das classes, o projeto implementa uma busca por divisões candidatas. A configuração `group_candidates: 512` permite avaliar até 512 possibilidades.

Cada candidata é analisada considerando:

- A proximidade entre o tamanho real de cada conjunto e as proporções desejadas.
- A distribuição das categorias em cada conjunto em comparação com a distribuição geral do dataset.
- A presença de todas as classes nos três conjuntos, exigência ativada por `require_all_classes: true`.

A configuração `max_size_deviation: 0.1` permite uma diferença máxima de 10 pontos percentuais entre a proporção desejada e a proporção obtida em cada conjunto.

Entre as candidatas que atendem aos critérios, o algoritmo seleciona aquela com a menor pontuação de desvio. Caso nenhuma divisão satisfaça as condições, o processo é interrompido com um erro.

### Verificações e auditoria

Após definir as partições, o código executa verificações para garantir a consistência da divisão:

- Confere se os três conjuntos possuem registros.
- Verifica se todas as categorias estão presentes em cada conjunto.
- Confere se os tamanhos respeitam a tolerância configurada.
- Verifica se um mesmo grupo de imagens não aparece em partições diferentes.
- Audita possíveis valores compartilhados entre os conjuntos, incluindo os hashes das imagens.

A configuração `overlap_policy: warn` determina que eventuais sobreposições identificadas pela auditoria sejam registradas como avisos, em vez de interromper automaticamente o processo. A separação dos grupos de imagens, entretanto, também é verificada diretamente, e uma violação nessa regra gera erro.

### Manifesto e reprodutibilidade

Para manter a consistência entre as etapas do projeto, o módulo gera um manifesto JSON em `data/processed/splits/chamados.json`.

Esse arquivo registra:

- O identificador de cada chamado e sua respectiva partição.
- A quantidade de registros e a proporção de cada conjunto.
- A distribuição das classes e a quantidade de grupos.
- Informações de configuração e hashes utilizados para verificar a integridade dos dados e das atribuições.

O código também utiliza `seed: 42` para tornar a geração da divisão reproduzível.

Quando um manifesto válido já existe e corresponde ao dataset e à configuração atuais, o sistema reutiliza a divisão anterior. Se os dados ou as configurações forem alterados, é necessário recriar o manifesto por meio da opção `--rebuild-split`.

### Importância para o Machine Learning

A separação dos dados é fundamental para avaliar modelos de maneira confiável. O conjunto de treinamento será utilizado para o aprendizado, enquanto a validação permitirá acompanhar e comparar os modelos durante o desenvolvimento. O conjunto de teste ficará reservado para a avaliação final.

Além disso, o agrupamento por conteúdo das imagens reduz a possibilidade de informações praticamente idênticas aparecerem simultaneamente no treinamento e na avaliação, contribuindo para uma análise mais confiável da capacidade de generalização dos modelos.

## 7) Integração do pipeline — `pipeline.py`

O módulo `pipeline.py` funciona como ponto central de execução do fluxo de dados do projeto. Sua responsabilidade é coordenar as etapas de processamento, validação e divisão do dataset, além de permitir a execução de etapas adicionais por meio de argumentos opcionais.

### Fluxo principal de execução

Na execução padrão, o pipeline realiza duas etapas principais:

1. **Processamento e validação (`processor.py`):** carrega o arquivo de entrada definido em `sources.yaml`, aplica as conversões de tipos, remove as colunas configuradas, executa as verificações de qualidade e salva o resultado em formato Parquet.
2. **Divisão dos dados (`splitter.py`):** utiliza o dataset processado para criar uma nova divisão entre treinamento, validação e teste ou reutilizar uma divisão existente, conforme as configurações de `models.yaml`.

O fluxo pode ser representado da seguinte forma:

`chamados-clean.csv` → `processor.py` → `chamados.parquet` → `splitter.py` → `chamados.json`

O arquivo Parquet armazena o dataset processado, enquanto o manifesto JSON registra a divisão dos registros entre os conjuntos.

### Configurações e integração entre módulos

O pipeline utiliza dois arquivos de configuração:

- **`sources.yaml`:** define os caminhos dos arquivos de entrada e saída, as conversões de tipos, as colunas que devem ser removidas e os critérios de qualidade dos dados.
- **`models.yaml`:** define o dataset utilizado pelas etapas posteriores, as configurações da divisão dos dados e as opções relacionadas aos geradores de características e à classificação multimodal.

Antes de iniciar o processamento, o pipeline verifica se o caminho de saída definido em `sources.yaml` corresponde ao dataset configurado em `models.yaml`. Essa verificação evita que as etapas trabalhem com arquivos diferentes.

Também são verificadas as configurações necessárias para as etapas opcionais, conforme os argumentos informados na execução.

### Modos de execução

O pipeline utiliza argumentos de linha de comando para controlar quais etapas serão executadas.

Os principais comandos são:

| Comando | Finalidade |
|---|---|
| `python -m src.data.pipeline` | Processa, valida e salva o dataset, além de criar ou reutilizar a divisão dos dados. |
| `python -m src.data.pipeline --dry-run` | Simula o processamento e as verificações, sem salvar o dataset final ou baixar modelos. |
| `python -m src.data.pipeline --skip-split` | Executa o processamento e a validação sem criar ou reutilizar a divisão dos dados. |
| `python -m src.data.pipeline --rebuild-split` | Recria deliberadamente a divisão dos dados e seu manifesto. |

O módulo também oferece argumentos opcionais para etapas posteriores, como seleção de características, preparação de modelos e execução do classificador multimodal. Essas funcionalidades pertencem às etapas de desenvolvimento e treinamento dos modelos, não sendo detalhadas nesta documentação.

Algumas combinações de argumentos são bloqueadas pelo próprio pipeline para evitar execuções incompatíveis, como utilizar `--skip-split` e `--rebuild-split` simultaneamente.

### Tratamento de erros e controle do fluxo

Durante a execução, o pipeline coordena os módulos e interrompe o fluxo quando uma etapa essencial apresenta falha.

Por exemplo, se o processamento ou a validação do dataset falhar, a divisão dos dados não será executada. Da mesma forma, erros relacionados a configurações, arquivos ausentes ou dependências são capturados e apresentados como mensagens de erro.

Essa organização centraliza a execução das etapas e reduz a necessidade de executar manualmente cada módulo de processamento e divisão.

### Resultado da integração

Com a integração dos módulos, o projeto estabelece um fluxo organizado de preparação dos dados:

- Os dados são processados e convertidos para um formato adequado às etapas posteriores.
- As regras de qualidade são verificadas antes da gravação do dataset processado.
- A divisão dos dados é criada ou reutilizada de maneira controlada e reproduzível.
- Os arquivos resultantes ficam disponíveis para as etapas posteriores de geração de características e Machine Learning.

Dessa forma, o pipeline conecta as atividades de limpeza, processamento, validação e divisão em um fluxo coordenado, documentado e configurável.